# CNN

In [ ]:
import os
from PIL import Image
import numpy as np
import glob
import random
import shutil
from pathlib import Path

import kagglehub
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19, ResNet50V2, InceptionV3
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import mixed_precision

### Load data

In [2]:
# Download dataset
path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
print(f"Dataset downloaded to: {path}")

Using Colab cache for faster access to the 'tiny-imagenet' dataset.
Dataset downloaded to: /kaggle/input/tiny-imagenet

Contents:
['tiny-imagenet-200']

/kaggle/input/tiny-imagenet:
  Directories: ['tiny-imagenet-200']
  Files: []


In [4]:
image_paths = glob.glob(f"/kaggle/input/tiny-imagenet/tiny-imagenet-200/train/n01443537/images/*.JPEG")
print(f"Found {len(image_paths)} images")

# Load first image
img = Image.open(image_paths[0])
img_array = np.array(img)
print(f"Image shape: {img_array.shape}")

Found 500 images
Image shape: (64, 64, 3)


In [ ]:
# Enable mixed precision for faster training on Colab
mixed_precision.set_global_policy('mixed_float16')

# -------------------------------
# Settings
# -------------------------------
train_dir = '/kaggle/input/tiny-imagenet/tiny-imagenet-200/train'
num_classes = 200
images_per_class = 25
batch_size = 64
epochs = 2

# -------------------------------
# Create subset
# -------------------------------

def create_physical_subset(train_dir, output_dir, images_per_class=15):
    """Creates a smaller dataset directory for faster loading"""
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    for class_name in os.listdir(train_dir):
        src_class = os.path.join(train_dir, class_name, 'images')
        dst_class = os.path.join(output_dir, class_name)

        if os.path.exists(src_class):
            os.makedirs(dst_class, exist_ok=True)
            imgs = os.listdir(src_class)
            selected = random.sample(imgs, min(len(imgs), images_per_class))

            for img in selected:
                shutil.copy2(os.path.join(src_class, img),
                           os.path.join(dst_class, img))

    print(f"Created subset with {images_per_class} images per class")

# Create subset directory
subset_dir = '/tmp/tiny_imagenet_subset'
create_physical_subset(train_dir, subset_dir, images_per_class)

# -------------------------------
# Generators
# -------------------------------
def make_generator(base_size):

    train_datagen = ImageDataGenerator(
        rescale=1./255,
        horizontal_flip=True,
        validation_split=0.15  # 15% validation
    )

    train_gen = train_datagen.flow_from_directory(
        subset_dir,
        target_size=base_size,
        batch_size=batch_size,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )

    val_gen = train_datagen.flow_from_directory(
        subset_dir,
        target_size=base_size,
        batch_size=batch_size,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )

    return train_gen, val_gen

# -------------------------------
# Model builder
# -------------------------------
def build_model(base_model):
    # Freeze all layers
    base_model.trainable = False

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.3)(x)

    # Mixed precision requires explicit float32 for final layer
    outputs = Dense(num_classes, activation='softmax', dtype='float32')(x)

    model = Model(inputs=base_model.input, outputs=outputs)
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

In [8]:
# -------------------------------
# Model configurations
# -------------------------------
models_config = [
    ("VGG19", VGG19(weights='imagenet', include_top=False, input_shape=(224,224,3)), (224,224)),
    ("ResNet50V2", ResNet50V2(weights='imagenet', include_top=False, input_shape=(224,224,3)), (224,224)),
    ("InceptionV3", InceptionV3(weights='imagenet', include_top=False, input_shape=(224,224,3)), (224,224)),
]

# -------------------------------
# Train and compare
# -------------------------------
results = []

for name, base_model, size in models_config:
    print(f"\n{'='*50}")
    print(f"Training {name}")
    print(f"{'='*50}")

    train_gen, val_gen = make_generator(size)
    model = build_model(base_model)

    # Calculate steps
    steps_per_epoch = len(train_gen)
    validation_steps = len(val_gen)

    print(f"Steps per epoch: {steps_per_epoch}, Validation steps: {validation_steps}")

    # Train with early stopping
    es = EarlyStopping(monitor='val_accuracy', patience=1, restore_best_weights=True)

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=[es],
        verbose=1
    )

    val_acc = max(history.history['val_accuracy'])
    train_acc = max(history.history['accuracy'])
    results.append((name, train_acc, val_acc))

    # Clear memory
    del model, train_gen, val_gen
    import gc
    gc.collect()

# -------------------------------
# Results table
# -------------------------------
print("\n" + "="*60)
print("VALIDATION ACCURACY COMPARISON")
print("="*60)
for name, train_acc, val_acc in sorted(results, key=lambda x: x[2], reverse=True):
    print(name)
    print('Train acc:', train_acc)
    print('Val acc', val_acc)
    print()

Created subset with 25 images per class

Training VGG19
Found 4400 images belonging to 200 classes.
Found 600 images belonging to 200 classes.
Steps per epoch: 69, Validation steps: 10
Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


69/69 ━━━━━━━━━━━━━━━━━━━━ 18s 217ms/step - accuracy: 0.0043 - loss: 5.5145 - val_accuracy: 0.0183 - val_loss: 5.1930
Epoch 2/2
69/69 ━━━━━━━━━━━━━━━━━━━━ 11s 155ms/step - accuracy: 0.0128 - loss: 5.2529 - val_accuracy: 0.0433 - val_loss: 5.0830

Training ResNet50V2
Found 4400 images belonging to 200 classes.
Found 600 images belonging to 200 classes.
Steps per epoch: 69, Validation steps: 10
Epoch 1/2
69/69 ━━━━━━━━━━━━━━━━━━━━ 29s 280ms/step - accuracy: 0.0574 - loss: 5.4319 - val_accuracy: 0.3717 - val_loss: 2.9221
Epoch 2/2
69/69 ━━━━━━━━━━━━━━━━━━━━ 8s 116ms/step - accuracy: 0.4881 - loss: 2.2116 - val_accuracy: 0.4817 - val_loss: 2.2898

Training InceptionV3
Found 4400 images belonging to 200 classes.
Found 600 images belonging to 200 classes.
Steps per epoch: 69, Validation steps: 10
Epoch 1/2
69/69 ━━━━━━━━━━━━━━━━━━━━ 39s 376ms/step - accuracy: 0.0711 - loss: 5.4547 - val_accuracy: 0.4617 - val_loss: 2.7568
Epoch 2/2
69/69 ━━━━━━━━━━━━━━━━━━━━ 9s 126ms/step - accuracy: 0.5249 